In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as pp
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import scipy as sp
from ipywidgets import widgets
import os
from typing import Optional, List, Tuple
from functools import reduce
import datetime
dailies = os.path.realpath('../../dailies/')
%matplotlib widget
import _daily_reporters as dr


from io import StringIO

In [ ]:
def get_energy_components(outfile):
    line_query = ' Print df_mp2-f12-xg'
    line_query = ' Print'
    tensors = []
    values = []
    with open(outfile, 'r') as f:
        iterator = enumerate(f)
        for n, line in iterator:
            #if line_query in line:
            if line.startswith(line_query):
                if '::' in line:
                    tensor_name = line.split('::')[-1].rstrip()
                else:
                    tensor_name = line.split()[-1].rstrip()
                for _ in range(4): 
                    #print(n, line)
                    n, line = next(iterator)
                val = line.split()[-1]
                tensors.append(tensor_name)
                values.append(float(val))

    df = pd.DataFrame(dict(tensors=tensors, values=values))
    cval = 0
    for n, name in enumerate(df.tensors):
        if 'ET' in name:
            cval += df['values'][n]
    print( "Sum of all ET* and 1st EMp2F12s:")
    print(
        cval + df['values'][df.tensors == 'EMp2F12s[]'].values[0] 
    )
    return df    

In [ ]:
import generate_inputs_and_folders as giaf
import check_outputs_and_folders as coaf

In [ ]:

def get_all_df(gamma_set, args, meta, filters=None):
    outfiles = []
    dfs = []
    data = {}
    for f, folder, _, kwargs in giaf.generate_file_paths(args, meta):
        if filters is not None:
            skip = False
            for key, vals in filters.items():
                if kwargs[key] not in vals:
                    skip = True
                    break
            if skip: break

        gs = kwargs['gamma_set']
        if gs != gamma_set: continue
        outfile = coaf.get_outfile(*f)
        if outfile is None: continue
        outfiles.append(outfile)
        df = get_energy_components(outfile)
        if not data:
            data['tensors'] = df.tensors.values

        label = kwargs['bases']
        data[label] = df['values'].values
        
    return pd.DataFrame(data)

In [ ]:
giaf.generate_file_paths(args, meta)

In [ ]:
args = giaf.argparse.Namespace(
    metadata_path='ne_fully_correlated/debug_xg3/metadata.json'
    #metadata_path='ne_fully_correlated/debug_xg_frozen/metadata.json'
)
meta = giaf.read_metadata(
    args.metadata_path
)

for f, folder, _, kwargs in giaf.generate_file_paths(args, meta):
    print(*f, folder)
    for key, val in kwargs.items():
        print(f"{key:20s} = {val}")
    break

In [ ]:
gamma_set = '1.00_2.00_1.50'

filters = dict(
    bases = ['aug-cc-pVDZ','aug-cc-pVTZ','aug-cc-pVQZ', 'aug-cc-pV5Z']
)
dfmixed = get_all_df(gamma_set, args, meta, filters=filters)

print("\nGAMMA_SET:")
display(gamma_set)
dfmixed

In [ ]:
gamma_set = '2.00_2.00_2.00'

dftwos = get_all_df(gamma_set, args, meta, filters=filters)

print("\nGAMMA_SET:")
display(gamma_set)
display(dftwos)

df_diff = dfmixed.copy()
df_diff.iloc[:,1:] = dfmixed.iloc[:,1:] - dftwos.iloc[:,1:]
display('Diff between mixed and twos')
df_diff

In [ ]:
gamma_set = '1.00_1.00_1.00'

dfones = get_all_df(gamma_set, args, meta, filters=filters)

print("\nGAMMA_SET:")
display(gamma_set)
display(dfones)

df_diff = dfmixed.copy()
df_diff.iloc[:,1:] = dfmixed.iloc[:,1:] - dfones.iloc[:,1:]
display('diff between mixed and ones')
df_diff

In [ ]:
from importlib import reload
reload(dr)
dr.dump_daily(dftwos, fname='xg_ETB_breakdown_gamma_twos_fixed_CF')

In [ ]:
dr.dump_daily(dfmixed, fname='xg_ETB_breakdown_gamma_mixed_fixed_CF')

In [ ]:
dr.dump_daily(dfones, fname='xg_ETB_breakdown_gamma_ones_fixed_CF')

In [ ]:
dr.dump_daily(df_diff, fname='xg_energy_breakdwon_diff_between_ones_and_mixed_frozen')

In [ ]:
def get_et_sum(df):
    cval = 0
    emp2_sum = 0
    for n, name in enumerate(df.tensors):
        if 'ET' in name:
            cval += df['values'][n]
        if 'EMp2F12' in name:
            cval += df['values'][n]
            break
    return cval

for df in dfs:
    #display(df)
    print(get_et_sum(df), df.iloc[6])

## From Google sheets, after copying above

In [ ]:
csvdata = """gamma_set = 1.00_2.00_1.50 (mixed),,,,,,gamma_set = 2.00_2.00_2.00 (twos),,,,,,difference (twos - mixed),,,,
tensors,aug-cc-pVDZ,aug-cc-pVTZ,aug-cc-pVQZ,aug-cc-pV5Z,,tensors,aug-cc-pVDZ,aug-cc-pVTZ,aug-cc-pVQZ,aug-cc-pV5Z,,tensors,aug-cc-pVDZ,aug-cc-pVTZ,aug-cc-pVQZ,aug-cc-pV5Z
ERef,-128.496365,-128.533266,-128.543752,-128.54678,,ERef,-128.496365,-128.533266,-128.543752,-128.54678,,ERef,0.00000,0.00000,0.00000,0.00000
EDi2,-0.209048,-0.285825,-0.329979,-0.347945,,EDi2,-0.209048,-0.285825,-0.329979,-0.347945,,EDi2,0.00000,0.00000,0.00000,0.00000
EMp2F12s,0.003418,0.000844,0.001314,0.002171,,EMp2F12s,0.002946,0.001792,0.001642,0.002189,,EMp2F12s,-0.00047,0.00095,0.00033,0.00002
ETV,-0.441946,-0.257659,-0.148973,-0.105232,,ETV,-0.344806,-0.227973,-0.13994,-0.101619,,ETV,0.09714,0.02969,0.00903,0.00361
ETX,0.070278,0.043715,0.022094,0.014328,,ETX,0.048571,0.035139,0.019445,0.013125,,ETX,-0.02171,-0.00858,-0.00265,-0.00120
ETC,0.002989,-0.00007,-0.000277,0.001489,,ETC,0.002789,0.001343,0.000631,0.001642,,ETC,-0.00020,0.00141,0.00091,0.00015
ETB,0.246344,0.100807,-0.513738,-1.129527,,ETB,0.147397,0.109826,0.074321,0.054667,,ETB,-0.09895,0.00902,0.58806,1.18419
EMp2F12s+,-0.118917,-0.112362,-0.639581,-1.216771,,EMp2F12s+,-0.143104,-0.079874,-0.043901,-0.029997,,EMp2F12s+,-0.02419,0.03249,0.59568,1.18677
corr_e,-0.327965,-0.398188,-0.969559,-1.564716,,corr_e,-0.352151,-0.365698,-0.37388,-0.377941,,corr_e,-0.02419,0.03249,0.59568,1.18678
check Emp2f12,-0.118917,-0.112363,-0.63958,-1.216771,,check Emp2f12,-0.143103,-0.079873,-0.043901,-0.029996,,check Emp2f12,-0.02419,0.03249,0.59568,1.18678"""
                                                                                                                                                                                                               

In [ ]:
dr.dump_daily(csvdata, fname='xg_breakdown_analysis_google_sheets.csv')

In [ ]:
iofile = StringIO(csvdata)
pd.read_csv(